# Structured Outputs with Pydantic (CrewAI)

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app).

In [ ]:
!pip install -q crewai pydantic

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## `output_pydantic`: typed task output

Define a `BaseModel`, pass it as `output_pydantic` on the `Task`, run the crew, then read `result.pydantic`.

In [ ]:
from pydantic import BaseModel, Field

from crewai import Agent, Crew, Process, Task


class Section(BaseModel):
    heading: str
    body: str


class BlogPost(BaseModel):
    title: str
    content: str
    tags: list[str]
    sections: list[Section] = Field(default_factory=list)


writer = Agent(
    role="Technical Writer",
    goal="Produce accurate, structured blog posts",
    backstory="You always return output that matches the requested schema.",
    verbose=True,
)

task = Task(
    description=(
        "Write a short blog post about AI agents. Include at least two sections with headings "
        "and bodies, and at least two tags."
    ),
    expected_output="A blog post with title, content, tags, and sections per the Pydantic model.",
    agent=writer,
    output_pydantic=BlogPost,
)

crew = Crew(
    agents=[writer],
    tasks=[task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff()
print(result.pydantic.title)
print(result.pydantic.tags)
for s in result.pydantic.sections:
    print(s.heading, "->", s.body[:80], "...")

## `output_json`: dict instead of a model

Same schema, but use `output_json=BlogPost` and read `result.json_dict`.

In [ ]:
task_json = Task(
    description="Write a very short blog post about CrewAI structured outputs (one paragraph).",
    expected_output="Title, content, and tags as JSON-shaped fields.",
    agent=writer,
    output_json=BlogPost,
)

crew_json = Crew(
    agents=[writer],
    tasks=[task_json],
    process=Process.sequential,
    verbose=True,
)

result_json = crew_json.kickoff()
print(result_json.json_dict["title"])
print(result_json.json_dict.get("tags", []))

## Key takeaways

- **`output_pydantic`**: validated `BaseModel` on `result.pydantic` — type hints, nesting, and `.model_dump()`.
- **`output_json`**: same schema, plain **`dict`** on `result.json_dict` — good when you only need key lookups or JSON.
- **`expected_output`** still documents quality for the agent; the Pydantic model defines **shape**.
- Prefer **Pydantic** in Python pipelines; use **JSON dict** when a dynamic mapping is enough.